# 05 – Grouped feature importance and subgroup (fairness) check
Permutation importance of single correlated features understates their value (shuffling one
feature leaves its correlated twins intact). Here whole GROUPS of related features are shuffled together.
Then the final model is checked for equal at-risk detection across student groups.

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score
from common import *

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
model = joblib.load(MODEL_DIR / "student_performance_model.joblib")
test = pd.read_csv(ROOT / "data" / "processed" / "test.csv", low_memory=False)
feats, num, cat = split_columns(test, True)
classes = list(model.named_steps["model"].classes_)

## 1. Grouped permutation importance (macro-F1 drop, 3 repeats, 15,000 test rows)

In [ ]:
GROUPS = {
    "Attendance": ["attendance_rate", "absence_days", "attendance_x_submission"],
    "Assessment scores": ["midterm_score", "quiz_avg_score", "assignment_avg_score", "avg_assessment_score", "midterm_vs_coursework"],
    "Assignment behaviour": ["assignment_submission_rate", "late_submissions"],
    "Online engagement": ["lms_logins_week", "video_completion_rate", "online_engagement", "forum_posts"],
    "Study effort": ["study_hours_week", "study_per_credit", "library_visits_month", "support_sessions"],
    "Prior academics": ["prior_gpa", "entry_exam_score", "academic_gap"],
    "Course context": ["course_difficulty", "course_load_credits", "program", "semester", "academic_year", "year_of_study", "study_level"],
    "Personal / social": ["age", "employment_hours_week", "commute_minutes", "scholarship_status", "financial_aid", "enrollment_status",
                          "admission_type", "institution_type", "internet_access", "device_access", "low_connectivity", "total_weekly_load"],
}
used = {c for g in GROUPS.values() for c in g}
assert used <= set(feats), used - set(feats)
ts = test.sample(15000, random_state=RANDOM_STATE).reset_index(drop=True)
base = f1_score(ts[TARGET], model.predict(ts[feats]), average="macro")
rng = np.random.RandomState(RANDOM_STATE)
rows = []
for g, cols in GROUPS.items():
    drops = []
    for _ in range(3):
        sh = ts.copy()
        idx = rng.permutation(len(sh))
        sh[cols] = ts[cols].iloc[idx].values            # shuffle the whole group together
        drops.append(base - f1_score(sh[TARGET], model.predict(sh[feats]), average="macro"))
    rows.append({"group": g, "n_features": len(cols), "macro_f1_drop": np.mean(drops), "std": np.std(drops)})
gi = pd.DataFrame(rows).sort_values("macro_f1_drop", ascending=False)
gi.to_csv(RES_DIR / "grouped_importance.csv", index=False); print(base, "\n", gi.round(4))

fig, ax = plt.subplots(figsize=(8, 4.4)); g2 = gi.iloc[::-1]
ax.barh(g2.group, g2.macro_f1_drop, xerr=g2["std"], color="#2E6FA3")
ax.set_xlabel("Drop in macro-F1 when the whole group is shuffled"); ax.set_title("Which kinds of information drive the predictions?")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig14_grouped_importance.png"); plt.close()

## 2. Subgroup check: is at-risk detection similar for different student groups?

In [ ]:
p = model.predict_proba(test[feats])
test["p_risk"] = p[:, [classes.index(c) for c in AT_RISK]].sum(axis=1)
test["actual_risk"] = test[TARGET].isin(AT_RISK).astype(int)
test["flag"] = (test["p_risk"] >= 0.5).astype(int)
rows = []
for col in ["scholarship_status", "financial_aid", "enrollment_status", "study_level", "internet_access", "device_access",
            "institution_type", "admission_type", "program"]:
    for v, d in test.groupby(col, dropna=False):
        rows.append({"attribute": col, "group": "Missing" if pd.isna(v) else str(v), "n": len(d), "actual_at_risk_pct": d.actual_risk.mean() * 100,
                     "flagged_pct": d.flag.mean() * 100, "recall": recall_score(d.actual_risk, d.flag),
                     "precision": precision_score(d.actual_risk, d.flag)})
fair = pd.DataFrame(rows).round(3); fair.to_csv(RES_DIR / "fairness_subgroups.csv", index=False)
print(fair.to_string())
gap = fair.groupby("attribute")["recall"].agg(lambda s: s.max() - s.min()).round(3)
gap.to_csv(RES_DIR / "fairness_recall_gap.csv"); print(gap)

fig, ax = plt.subplots(figsize=(9, 5.4))
f2 = fair[fair.attribute.isin(["scholarship_status", "enrollment_status", "study_level", "internet_access", "device_access", "institution_type"])]
labels = (f2.attribute.str.replace("_", " ") + ": " + f2.group).tolist()
ax.barh(labels[::-1], f2.recall.values[::-1], color="#2E6FA3")
ax.axvline(recall_score(test.actual_risk, test.flag), color="#D1495B", ls="--", label="overall recall")
ax.set_xlim(0.4, 0.9); ax.set_xlabel("At-risk recall (threshold 0.5)"); ax.legend(frameon=False)
ax.set_title("At-risk detection rate by student group"); plt.tight_layout(); plt.savefig(FIG_DIR / "fig15_fairness_recall.png"); plt.close()